# IMTS Predictive Quality Analysis
## Field Concrete 28-Day Strength — 5-Fold Project-Grouped Cross-Validation

### Objective
Evaluate whether IMTS field concrete data contains useful predictive information for 28-day concrete strength.

We compare three levels of information:

1. **Day 0 — Field + Required Strength**  
   Information available on the day concrete is placed.

2. **Day 7 — Field + Required Strength + 7-Day Strength**  
   Adds early strength information available after seven days.

3. **Full Context + Day 7**  
   Adds historical **Supplier / Plant / Mix** context to the Day-7 information.

### Validation approach
- All three feature sets use the **same records with valid 7-day strength**.
- Cross-validation is **grouped by project**, so the same project is not in both training and validation within a fold.
- Supplier / Plant / Mix target encoding is created **inside each training fold** to reduce data leakage.
- Five regression models are compared: Dummy Mean, Ridge, Random Forest, HistGradientBoosting, and XGBoost.

## 1. Setup

This notebook is designed for Microsoft Fabric.

**Before running:**  
Attach the Lakehouse that contains the cleaned Field Core CSV and update `INPUT_PATH` below.

A common Fabric Lakehouse path looks like:

`/lakehouse/default/Files/.../field_core_clean_with_required.csv`

In [ ]:
# If XGBoost is not available in your Fabric environment,
# install it through the Fabric Environment or uncomment the next line.
# %pip install xgboost

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import math
import time
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor

print("Libraries loaded successfully.")

## 2. Configuration

The target is the average actual 28-day concrete strength.

The three feature sets are intentionally designed to answer three business questions:

| Feature Set | Business Question |
|---|---|
| Day 0 | Can IMTS identify strength risk on the day of placement? |
| Day 7 | How much does the 7-day result improve prediction? |
| Full Context + Day 7 | Does Supplier / Plant / Mix history add additional value? |

In [ ]:
# ---------------------------------------------------------------------
# Update this path to the CSV location in your attached Fabric Lakehouse.
# ---------------------------------------------------------------------
INPUT_PATH = (
    "/lakehouse/default/Files/field_core_outputs/"
    "field_core_clean/field_core_clean_with_required.csv"
)

TARGET = "AverageActualStrength28_psi"
REQUIRED_STRENGTH = "ApplicableSpecifiedStrength28"

RANDOM_STATE = 42
OUTER_CV_FOLDS = 5
TARGET_ENCODING_FOLDS = 5
TARGET_ENCODING_SMOOTHING = 20.0

FIELD_FEATURES = [
    "EffectiveSlump_in",
    "EffectiveAir_percent",
    "EffectiveUnitWeight_lb_ft3",
    "EffectiveConcreteTemp_F",
    "AmbientTemp_F",
    "WaterAdded_gal_per_yd3",
    "BatchToSampleMinutes",
    "BatchToCastMinutes",
    "HasWaterAdded",
    "HasAnyAfterSPMeasurement",
]

DAY0_FEATURES = FIELD_FEATURES + [REQUIRED_STRENGTH]

DAY7_AVERAGE_CANDIDATES = [
    "AverageActualStrength7_psi",
    "AverageActualStrength7",
]

DAY7_COUNT_CANDIDATES = [
    "ActualStrength7SpecimenCount",
    "StandardCuredStrength7SpecimenCount",
]

DAY7_FEATURES = [
    "Day7AverageStrength_psi",
    "Day7SpecimenCount",
    "Day7MarginToRequired_psi",
    "Day7ToRequiredRatio",
]

CONTEXT_COLUMN_CANDIDATES = {
    "Supplier": ["SupplierId", "supplierId", "SupplierName", "supplierName"],
    "Plant": ["PlantNumber", "plantNumber", "PlantNo", "plantNo"],
    "Mix": ["MixNumber", "mixNumber", "MixNo", "mixNo"],
}

GROUP_COLUMN_CANDIDATES = [
    "projectId",
    "projectNo",
    "ProjectId",
    "ProjectNo",
]

print("Configuration ready.")

## 3. Load the IMTS Field Core Dataset

This step loads the cleaned Field Core data.

The notebook will first show:
- total records,
- total columns,
- available project count,
- a small sample of the data.

In [ ]:
def read_csv(path: str) -> pd.DataFrame:
    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"Could not determine CSV encoding for: {path}")


df_raw = read_csv(INPUT_PATH)

project_col_preview = next(
    (c for c in GROUP_COLUMN_CANDIDATES if c in df_raw.columns),
    None,
)

dataset_overview = pd.DataFrame({
    "Metric": [
        "Total records",
        "Total columns",
        "Projects",
    ],
    "Value": [
        len(df_raw),
        df_raw.shape[1],
        df_raw[project_col_preview].nunique(dropna=True)
        if project_col_preview else np.nan,
    ],
})

display(dataset_overview)
display(df_raw.head(10))

## 4. Helper Functions

These functions prepare numeric values, project groups, Day-7 features, and Supplier / Plant / Mix context.

Missing project IDs are treated as unique test-level groups rather than being incorrectly combined into one project.

In [ ]:
def require_columns(df: pd.DataFrame, columns: Iterable[str]) -> None:
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise KeyError(f"Required columns are missing: {missing}")


def resolve_first_column(
    df: pd.DataFrame,
    candidates: Iterable[str],
) -> str | None:
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    return None


def numeric_series(df: pd.DataFrame, column: str) -> pd.Series:
    if column not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype=float)

    series = df[column]

    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    return pd.to_numeric(
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce",
    )


def numeric_frame(
    df: pd.DataFrame,
    columns: list[str],
) -> pd.DataFrame:
    require_columns(df, columns)
    result = pd.DataFrame(index=df.index)

    for column in columns:
        result[column] = numeric_series(df, column)

    return result


def resolve_group_column(df: pd.DataFrame) -> str:
    column = resolve_first_column(df, GROUP_COLUMN_CANDIDATES)

    if column is None:
        raise KeyError(
            "No project grouping column was found. "
            f"Expected one of: {GROUP_COLUMN_CANDIDATES}"
        )

    return column


def make_project_groups(
    df: pd.DataFrame,
    group_column: str,
) -> pd.Series:
    groups = df[group_column].astype("string").str.strip()
    missing = groups.isna() | groups.eq("")

    if "testId" in df.columns:
        fallback = "MISSING_PROJECT_TEST_" + df["testId"].astype("string")
    else:
        fallback = (
            "MISSING_PROJECT_ROW_"
            + pd.Series(df.index, index=df.index).astype("string")
        )

    return groups.mask(missing, fallback)

## 5. Create Day-7 Features and the Common Comparison Dataset

For a fair comparison, **all three feature sets are evaluated on the same records**.

Records must have:
- a valid 28-day actual strength,
- a valid required/design strength,
- a valid 7-day strength.

This prevents the Day-0 model from receiving an unfair advantage from having more records than the Day-7 models.

In [ ]:
def add_day7_features(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, str | None]]:
    result = df.copy()

    average_column = resolve_first_column(
        result,
        DAY7_AVERAGE_CANDIDATES,
    )

    count_column = resolve_first_column(
        result,
        DAY7_COUNT_CANDIDATES,
    )

    if average_column is None:
        raise KeyError(
            "No 7-day average-strength column was found. "
            f"Expected one of: {DAY7_AVERAGE_CANDIDATES}"
        )

    result["Day7AverageStrength_psi"] = numeric_series(
        result,
        average_column,
    )

    if count_column is None:
        result["Day7SpecimenCount"] = np.nan
    else:
        result["Day7SpecimenCount"] = numeric_series(
            result,
            count_column,
        )

    required = numeric_series(result, REQUIRED_STRENGTH)

    result["Day7MarginToRequired_psi"] = (
        result["Day7AverageStrength_psi"] - required
    )

    result["Day7ToRequiredRatio"] = (
        result["Day7AverageStrength_psi"]
        / required.replace(0, np.nan)
    )

    metadata = {
        "source_average_column": average_column,
        "source_count_column": count_column,
    }

    return result, metadata


require_columns(
    df_raw,
    [
        TARGET,
        REQUIRED_STRENGTH,
        *FIELD_FEATURES,
    ],
)

df = df_raw.copy()

target = numeric_series(df, TARGET)
required = numeric_series(df, REQUIRED_STRENGTH)

eligible_mask = target.gt(0) & required.gt(0)
eligible_count = int(eligible_mask.sum())

df = df.loc[eligible_mask].copy()
df, day7_metadata = add_day7_features(df)

valid_day7_mask = numeric_series(
    df,
    "Day7AverageStrength_psi",
).gt(0)

common_day7_count = int(valid_day7_mask.sum())
df = df.loc[valid_day7_mask].copy()

preparation_summary = pd.DataFrame({
    "Stage": [
        "Raw Field Core records",
        "Valid 28-day actual + required strength",
        "Common records with valid 7-day strength",
    ],
    "Records": [
        len(df_raw),
        eligible_count,
        common_day7_count,
    ],
})

display(preparation_summary)

print("Day-7 average source :", day7_metadata["source_average_column"])
print("Day-7 count source   :", day7_metadata["source_count_column"])

## 6. Project-Grouped Validation

The model is tested on **projects it did not train on within that fold**.

This is stricter and more realistic than randomly splitting individual concrete tests, because tests from the same project may share similar conditions.

### Concept

- Fold 1 → some projects train, different projects validate
- Fold 2 → a different project group validates
- ...
- Fold 5 → final project group validates

No project is allowed to appear in both training and validation within the same fold.

In [ ]:
group_column = resolve_group_column(df)
groups = make_project_groups(df, group_column)

unique_group_count = int(groups.nunique())

if unique_group_count < OUTER_CV_FOLDS:
    raise ValueError(
        f"Only {unique_group_count} project groups are available, "
        f"but OUTER_CV_FOLDS={OUTER_CV_FOLDS}."
    )

grouping_summary = pd.DataFrame({
    "Metric": [
        "Grouping column",
        "Common records used",
        "Unique project groups",
        "Outer CV folds",
    ],
    "Value": [
        group_column,
        len(df),
        unique_group_count,
        OUTER_CV_FOLDS,
    ],
})

display(grouping_summary)

## 7. Supplier / Plant / Mix Historical Context

The Full Context model adds historical information for:

- Supplier
- Plant
- Mix
- Supplier + Plant
- Supplier + Plant + Mix

### Leakage protection

The historical target encoding is calculated **only from training data**.

A validation record never uses its own 28-day strength to create its Supplier / Plant / Mix historical feature.

In [ ]:
@dataclass(frozen=True)
class ContextSources:
    supplier: str
    plant: str
    mix: str


def normalize_category(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .fillna("__MISSING__")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
        .replace("", "__MISSING__")
    )


def resolve_context_sources(df: pd.DataFrame) -> ContextSources:
    resolved: dict[str, str] = {}

    for logical_name, candidates in CONTEXT_COLUMN_CANDIDATES.items():
        column = resolve_first_column(df, candidates)

        if column is None:
            raise KeyError(
                f"Context field '{logical_name}' was not found. "
                f"Expected one of: {candidates}"
            )

        resolved[logical_name] = column

    return ContextSources(
        supplier=resolved["Supplier"],
        plant=resolved["Plant"],
        mix=resolved["Mix"],
    )


def build_context_categories(
    df: pd.DataFrame,
    sources: ContextSources,
) -> pd.DataFrame:
    context = pd.DataFrame(index=df.index)

    context["SupplierCategory"] = normalize_category(df[sources.supplier])
    context["PlantCategory"] = normalize_category(df[sources.plant])
    context["MixCategory"] = normalize_category(df[sources.mix])

    context["SupplierPlantCategory"] = (
        context["SupplierCategory"]
        + "|"
        + context["PlantCategory"]
    )

    context["SupplierPlantMixCategory"] = (
        context["SupplierCategory"]
        + "|"
        + context["PlantCategory"]
        + "|"
        + context["MixCategory"]
    )

    return context


def smoothed_target_mapping(
    category: pd.Series,
    target: pd.Series,
    global_mean: float,
    smoothing: float,
) -> pd.Series:
    stats = pd.DataFrame(
        {
            "category": category,
            "target": target,
        }
    ).groupby(
        "category",
        dropna=False,
    )["target"].agg(["sum", "count"])

    return (
        stats["sum"] + smoothing * global_mean
    ) / (
        stats["count"] + smoothing
    )


def cross_fitted_target_encode(
    train_categories: pd.DataFrame,
    validation_categories: pd.DataFrame,
    y_train: pd.Series,
    train_groups: pd.Series,
    *,
    smoothing: float = TARGET_ENCODING_SMOOTHING,
    max_splits: int = TARGET_ENCODING_FOLDS,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    y_train = pd.to_numeric(y_train, errors="coerce")

    if y_train.isna().any():
        raise ValueError("Target encoding received missing training targets.")

    groups_inner = (
        train_groups.astype("string")
        .fillna("__MISSING_GROUP__")
    )

    unique_groups = int(groups_inner.nunique())
    n_splits = min(max_splits, unique_groups)

    if n_splits < 2:
        raise ValueError(
            "At least two project groups are required "
            "for cross-fitted target encoding."
        )

    inner_splitter = GroupKFold(n_splits=n_splits)
    global_mean = float(y_train.mean())

    encoded_train = pd.DataFrame(index=train_categories.index)
    encoded_validation = pd.DataFrame(index=validation_categories.index)

    metadata_rows = []

    for column in train_categories.columns:
        train_values = normalize_category(train_categories[column])
        validation_values = normalize_category(validation_categories[column])

        oof = pd.Series(
            np.nan,
            index=train_categories.index,
            dtype=float,
        )

        for fit_positions, encoding_positions in inner_splitter.split(
            train_categories,
            y_train,
            groups_inner,
        ):
            fit_index = train_categories.index[fit_positions]
            encoding_index = train_categories.index[encoding_positions]

            mapping = smoothed_target_mapping(
                train_values.loc[fit_index],
                y_train.loc[fit_index],
                global_mean,
                smoothing,
            )

            oof.loc[encoding_index] = (
                train_values.loc[encoding_index]
                .map(mapping)
                .fillna(global_mean)
            )

        full_train_mapping = smoothed_target_mapping(
            train_values,
            y_train,
            global_mean,
            smoothing,
        )

        train_counts = train_values.value_counts(dropna=False)

        encoded_train[f"{column}_TargetMean"] = oof.fillna(global_mean)

        encoded_validation[f"{column}_TargetMean"] = (
            validation_values
            .map(full_train_mapping)
            .fillna(global_mean)
        )

        encoded_train[f"{column}_LogCount"] = np.log1p(
            train_values
            .map(train_counts)
            .fillna(0)
            .astype(float)
        )

        encoded_validation[f"{column}_LogCount"] = np.log1p(
            validation_values
            .map(train_counts)
            .fillna(0)
            .astype(float)
        )

        unknown_validation = ~validation_values.isin(
            full_train_mapping.index
        )

        encoded_train[f"{column}_Unknown"] = 0
        encoded_validation[f"{column}_Unknown"] = (
            unknown_validation.astype(int)
        )

        metadata_rows.append({
            "ContextColumn": column,
            "TrainUniqueCategories": int(train_values.nunique()),
            "ValidationUniqueCategories": int(validation_values.nunique()),
            "UnknownValidationRows": int(unknown_validation.sum()),
            "UnknownValidationPercent": float(
                unknown_validation.mean() * 100.0
            ),
            "InnerEncodingFolds": n_splits,
            "Smoothing": smoothing,
        })

    return (
        encoded_train,
        encoded_validation,
        pd.DataFrame(metadata_rows),
    )


context_sources = resolve_context_sources(df)

display(pd.DataFrame({
    "Context": ["Supplier", "Plant", "Mix"],
    "Source Column": [
        context_sources.supplier,
        context_sources.plant,
        context_sources.mix,
    ],
}))

## 8. Candidate Regression Models

We compare several model families rather than relying on one algorithm.

The purpose at this stage is to determine which model family is strongest and most stable under cross-validation, not to perform aggressive hyperparameter tuning.

In [ ]:
def build_regression_models() -> dict[str, object]:
    return {
        "DummyMean": DummyRegressor(
            strategy="mean",
        ),

        "Ridge": Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True,
                    ),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "model",
                    Ridge(alpha=10.0),
                ),
            ]
        ),

        "RandomForest": Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True,
                    ),
                ),
                (
                    "model",
                    RandomForestRegressor(
                        n_estimators=300,
                        min_samples_leaf=5,
                        max_features=0.8,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),

        "HistGradientBoosting": Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True,
                    ),
                ),
                (
                    "model",
                    HistGradientBoostingRegressor(
                        max_iter=300,
                        learning_rate=0.05,
                        max_leaf_nodes=31,
                        min_samples_leaf=20,
                        l2_regularization=1.0,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),

        "XGBoost": XGBRegressor(
            objective="reg:squarederror",
            n_estimators=700,
            learning_rate=0.04,
            max_depth=6,
            min_child_weight=5,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.0,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
            eval_metric="rmse",
        ),
    }


model_table = pd.DataFrame({
    "Model": list(build_regression_models().keys()),
    "Purpose": [
        "Simple baseline",
        "Linear regularized baseline",
        "Tree ensemble",
        "Gradient boosting",
        "Boosted tree model",
    ],
})

display(model_table)

## 9. Evaluation Metrics

The main metrics are:

- **MAE** — average absolute prediction error in psi. Lower is better.
- **RMSE** — gives more weight to larger prediction errors. Lower is better.
- **R²** — how much variation in 28-day strength the model explains. Higher is better.
- **Within 300 psi / 500 psi** — percentage of predictions close to the actual result.
- **Mean Bias** — whether predictions tend to run high or low.

In [ ]:
def regression_metrics(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> dict[str, float]:
    residual = predicted - actual
    absolute_error = np.abs(residual)

    return {
        "MAE": float(mean_absolute_error(actual, predicted)),
        "MedianAE": float(median_absolute_error(actual, predicted)),
        "RMSE": float(
            math.sqrt(mean_squared_error(actual, predicted))
        ),
        "R2": float(r2_score(actual, predicted)),
        "MeanBias": float(np.mean(residual)),
        "Within300PsiPercent": float(
            np.mean(absolute_error <= 300.0) * 100.0
        ),
        "Within500PsiPercent": float(
            np.mean(absolute_error <= 500.0) * 100.0
        ),
    }

print("Metric functions ready.")

# 10. Run 5-Fold Project-Grouped Cross-Validation

This is the main analysis.

For each fold:

1. Projects are separated into training and validation.
2. The notebook checks that no project overlaps.
3. The three feature sets are built.
4. Supplier / Plant / Mix encoding is learned from training data only.
5. Every regression model is trained and evaluated.

This cell may take several minutes depending on Fabric capacity and cluster resources.

In [ ]:
outer_splitter = GroupKFold(n_splits=OUTER_CV_FOLDS)

fold_metric_rows = []
prediction_frames = []
context_metadata_frames = []
fold_summary_rows = []

for fold_number, (train_positions, validation_positions) in enumerate(
    outer_splitter.split(df, groups=groups),
    start=1,
):
    print(f"Starting fold {fold_number}/{OUTER_CV_FOLDS}...")

    train = df.iloc[train_positions].copy()
    validation = df.iloc[validation_positions].copy()

    train_groups = make_project_groups(train, group_column)
    validation_groups = make_project_groups(validation, group_column)

    overlap = set(train_groups.astype(str)).intersection(
        set(validation_groups.astype(str))
    )

    if overlap:
        raise RuntimeError(
            f"Fold {fold_number}: project leakage detected "
            f"({len(overlap)} overlapping groups)."
        )

    y_train = numeric_series(train, TARGET)
    y_validation = numeric_series(validation, TARGET)

    fold_summary_rows.append({
        "Fold": fold_number,
        "TrainRows": len(train),
        "ValidationRows": len(validation),
        "TrainProjectGroups": int(train_groups.nunique()),
        "ValidationProjectGroups": int(validation_groups.nunique()),
        "TrainTargetMean": float(y_train.mean()),
        "TrainTargetStd": float(y_train.std()),
        "ValidationTargetMean": float(y_validation.mean()),
        "ValidationTargetStd": float(y_validation.std()),
        "ProjectOverlap": len(overlap),
    })

    # -------------------------------------------------------------
    # Feature Set 1: Day-0 Field + Required
    # -------------------------------------------------------------
    day0_train = numeric_frame(train, DAY0_FEATURES)
    day0_validation = numeric_frame(validation, DAY0_FEATURES)

    # -------------------------------------------------------------
    # Feature Set 2: Day-0 + Day-7
    # -------------------------------------------------------------
    day7_train = numeric_frame(train, DAY7_FEATURES)
    day7_validation = numeric_frame(validation, DAY7_FEATURES)

    day7_full_train = pd.concat(
        [day0_train, day7_train],
        axis=1,
    )

    day7_full_validation = pd.concat(
        [day0_validation, day7_validation],
        axis=1,
    )

    # -------------------------------------------------------------
    # Feature Set 3: Day-0 + Context + Day-7
    # -------------------------------------------------------------
    train_context = build_context_categories(
        train,
        context_sources,
    )

    validation_context = build_context_categories(
        validation,
        context_sources,
    )

    (
        encoded_context_train,
        encoded_context_validation,
        context_metadata,
    ) = cross_fitted_target_encode(
        train_context,
        validation_context,
        y_train,
        train_groups,
    )

    context_metadata.insert(0, "OuterFold", fold_number)
    context_metadata_frames.append(context_metadata)

    full_train = pd.concat(
        [
            day0_train,
            encoded_context_train,
            day7_train,
        ],
        axis=1,
    )

    full_validation = pd.concat(
        [
            day0_validation,
            encoded_context_validation,
            day7_validation,
        ],
        axis=1,
    )

    feature_sets = {
        "Day0_FieldPlusRequired": (
            day0_train,
            day0_validation,
        ),
        "Day7_FieldPlusRequired": (
            day7_full_train,
            day7_full_validation,
        ),
        "Full_ContextPlusDay7": (
            full_train,
            full_validation,
        ),
    }

    for feature_set_name, (
        x_train,
        x_validation,
    ) in feature_sets.items():

        for model_name, model in build_regression_models().items():
            start = time.perf_counter()

            model.fit(
                x_train,
                y_train,
            )

            predicted = model.predict(
                x_validation
            )

            elapsed = time.perf_counter() - start

            metrics = regression_metrics(
                y_validation.to_numpy(),
                predicted,
            )

            fold_metric_rows.append({
                "Fold": fold_number,
                "FeatureSet": feature_set_name,
                "Model": model_name,
                "TrainRows": len(train),
                "ValidationRows": len(validation),
                "FeatureCount": x_train.shape[1],
                "TrainingSeconds": elapsed,
                **metrics,
            })

            prediction_frame = pd.DataFrame({
                "Fold": fold_number,
                "FeatureSet": feature_set_name,
                "Model": model_name,
                "ActualStrength28_psi": y_validation.to_numpy(),
                "PredictedStrength28_psi": predicted,
                "ResidualPsi": (
                    predicted - y_validation.to_numpy()
                ),
                "AbsoluteErrorPsi": np.abs(
                    predicted - y_validation.to_numpy()
                ),
            }, index=validation.index)

            for identifier in [
                "testId",
                "projectId",
                "projectNo",
                "officeId",
                "OfficeName",
            ]:
                if identifier in validation.columns:
                    prediction_frame[identifier] = (
                        validation[identifier]
                        .reindex(prediction_frame.index)
                        .to_numpy()
                    )

            prediction_frames.append(
                prediction_frame.reset_index(drop=True)
            )

    print(f"Completed fold {fold_number}/{OUTER_CV_FOLDS}.")

print("Cross-validation complete.")

## 11. Verify the Project Splits

Every fold should show **ProjectOverlap = 0**.

This confirms that the same project did not appear in both training and validation within a fold.

In [ ]:
fold_summary = pd.DataFrame(fold_summary_rows)

display(
    fold_summary[
        [
            "Fold",
            "TrainRows",
            "ValidationRows",
            "TrainProjectGroups",
            "ValidationProjectGroups",
            "ProjectOverlap",
        ]
    ]
)

assert (fold_summary["ProjectOverlap"] == 0).all()

print("PASS: No project overlap was detected in any outer fold.")

## 12. Cross-Validation Results

The table below averages the five validation folds.

**Primary comparison:**  
Use **MAE** first, with RMSE and R² as supporting metrics.

In [ ]:
fold_metrics = pd.DataFrame(fold_metric_rows)

predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)

context_metadata_all = pd.concat(
    context_metadata_frames,
    ignore_index=True,
)

summary = (
    fold_metrics.groupby(
        ["FeatureSet", "Model"],
        as_index=False,
    )
    .agg(
        MeanCV_MAE=("MAE", "mean"),
        StdCV_MAE=("MAE", "std"),
        MeanCV_RMSE=("RMSE", "mean"),
        StdCV_RMSE=("RMSE", "std"),
        MeanCV_R2=("R2", "mean"),
        StdCV_R2=("R2", "std"),
        MeanCV_MedianAE=("MedianAE", "mean"),
        MeanCV_Bias=("MeanBias", "mean"),
        MeanWithin300PsiPercent=(
            "Within300PsiPercent",
            "mean",
        ),
        MeanWithin500PsiPercent=(
            "Within500PsiPercent",
            "mean",
        ),
        MeanTrainingSeconds=(
            "TrainingSeconds",
            "mean",
        ),
    )
    .sort_values(
        ["MeanCV_MAE", "MeanCV_RMSE"],
        ascending=True,
    )
    .reset_index(drop=True)
)

summary_display = summary.copy()

for col in [
    "MeanCV_MAE",
    "StdCV_MAE",
    "MeanCV_RMSE",
    "StdCV_RMSE",
    "MeanCV_MedianAE",
    "MeanCV_Bias",
    "MeanWithin300PsiPercent",
    "MeanWithin500PsiPercent",
    "MeanTrainingSeconds",
]:
    summary_display[col] = summary_display[col].round(1)

summary_display["MeanCV_R2"] = summary_display["MeanCV_R2"].round(3)
summary_display["StdCV_R2"] = summary_display["StdCV_R2"].round(3)

display(summary_display)

## 13. Best Model for Each Feature Set

This is the most useful summary for a business audience.

Rather than comparing every model, this table answers:

- What is the best Day-0 result?
- What is the best Day-7 result?
- What is the best Full Context result?

In [ ]:
best_by_feature_set = (
    summary.sort_values(
        ["MeanCV_MAE", "MeanCV_RMSE"],
        ascending=True,
    )
    .groupby(
        "FeatureSet",
        as_index=False,
    )
    .first()
    .sort_values(
        "MeanCV_MAE",
        ascending=True,
    )
    .reset_index(drop=True)
)

best_display = best_by_feature_set[
    [
        "FeatureSet",
        "Model",
        "MeanCV_MAE",
        "StdCV_MAE",
        "MeanCV_RMSE",
        "MeanCV_R2",
        "MeanWithin300PsiPercent",
        "MeanWithin500PsiPercent",
    ]
].copy()

best_display["MeanCV_MAE"] = best_display["MeanCV_MAE"].round(1)
best_display["StdCV_MAE"] = best_display["StdCV_MAE"].round(1)
best_display["MeanCV_RMSE"] = best_display["MeanCV_RMSE"].round(1)
best_display["MeanCV_R2"] = best_display["MeanCV_R2"].round(3)
best_display["MeanWithin300PsiPercent"] = (
    best_display["MeanWithin300PsiPercent"].round(1)
)
best_display["MeanWithin500PsiPercent"] = (
    best_display["MeanWithin500PsiPercent"].round(1)
)

display(best_display)

## 14. Visual Comparison — Prediction Error

Lower MAE is better.

This chart shows how much the prediction improves as more information becomes available.

In [ ]:
plot_data = best_by_feature_set.copy()

feature_order = [
    "Day0_FieldPlusRequired",
    "Day7_FieldPlusRequired",
    "Full_ContextPlusDay7",
]

plot_data = (
    plot_data
    .set_index("FeatureSet")
    .reindex(feature_order)
    .reset_index()
)

labels = [
    "Day 0",
    "Day 7",
    "Full Context + Day 7",
]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, plot_data["MeanCV_MAE"])

ax.set_title("Best Cross-Validated MAE by Information Available")
ax.set_ylabel("Mean Absolute Error (psi)")
ax.set_xlabel("Feature Set")
ax.grid(axis="y", alpha=0.25)

for bar, value in zip(bars, plot_data["MeanCV_MAE"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.0f} psi",
        ha="center",
        va="bottom",
    )

plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 15. Visual Comparison — Explained Variation

Higher R² is better.

This chart shows how much of the variation in 28-day concrete strength is explained by the best model at each information stage.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, plot_data["MeanCV_R2"])

ax.set_title("Best Cross-Validated R² by Information Available")
ax.set_ylabel("Mean R²")
ax.set_xlabel("Feature Set")
ax.grid(axis="y", alpha=0.25)

for bar, value in zip(bars, plot_data["MeanCV_R2"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.3f}",
        ha="center",
        va="bottom",
    )

plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 16. Supplier / Plant / Mix Coverage in Validation Data

This table shows how often a Supplier / Plant / Mix context appearing in validation was not present in the corresponding training data.

A high unknown percentage means the model often encounters new context that it has not previously seen.

In [ ]:
context_summary = (
    context_metadata_all
    .groupby("ContextColumn", as_index=False)
    .agg(
        MeanUnknownValidationPercent=(
            "UnknownValidationPercent",
            "mean",
        ),
        MaxUnknownValidationPercent=(
            "UnknownValidationPercent",
            "max",
        ),
        MeanTrainUniqueCategories=(
            "TrainUniqueCategories",
            "mean",
        ),
    )
)

context_summary["MeanUnknownValidationPercent"] = (
    context_summary["MeanUnknownValidationPercent"].round(1)
)
context_summary["MaxUnknownValidationPercent"] = (
    context_summary["MaxUnknownValidationPercent"].round(1)
)
context_summary["MeanTrainUniqueCategories"] = (
    context_summary["MeanTrainUniqueCategories"].round(0)
)

display(context_summary)

# 17. Business Interpretation

### What the three comparisons mean

**Day 0**  
Shows whether information already available when concrete is placed can provide an early indication of future 28-day strength.

**Day 7**  
Shows how much prediction improves after early strength becomes available.

**Full Context + Day 7**  
Shows whether IMTS historical Supplier / Plant / Mix patterns provide additional information beyond the individual test measurements.

### Important business consideration

A more accurate Day-7 model does not automatically mean it is the most valuable user-facing prediction.

Experienced project managers can already interpret a 7-day strength result to some extent. Therefore, the strongest business value may come from:

- identifying elevated risk **before the 7-day result is available**, or
- combining the 7-day result with historical patterns that are difficult for a person to evaluate manually.

The model metrics should therefore be considered together with **when the prediction becomes available** and **what decision the user can still make at that time**.

## 18. Final Conclusion Template

After reviewing the actual results above, update the wording below if necessary.

### Suggested executive summary

> The project-grouped cross-validation results show whether IMTS concrete test data contains repeatable predictive information across different projects.  
>
> The Day-0 model measures the value of information available at placement, while the Day-7 comparison measures the improvement from early strength results. The Full Context model evaluates whether historical Supplier / Plant / Mix information adds further value.  
>
> Because projects are separated between training and validation and historical context is created from training data only, the results are designed to represent a more realistic test of model performance on unseen project data.  
>
> The business value should be judged not only by model accuracy, but also by how early the prediction is available and whether it provides information that a project manager could not easily determine from the raw test results alone.

## 19. Optional — Save Results to the Fabric Lakehouse

Uncomment and update `OUTPUT_DIR` if you want to keep the analysis outputs as CSV files.

In [ ]:
# OUTPUT_DIR = (
#     "/lakehouse/default/Files/field_core_outputs/"
#     "consolidated_three_model_cross_validation"
# )
#
# import os
# os.makedirs(OUTPUT_DIR, exist_ok=True)
#
# fold_metrics.to_csv(
#     f"{OUTPUT_DIR}/cv_fold_metrics.csv",
#     index=False,
# )
#
# summary.to_csv(
#     f"{OUTPUT_DIR}/cv_summary.csv",
#     index=False,
# )
#
# best_by_feature_set.to_csv(
#     f"{OUTPUT_DIR}/best_model_by_feature_set_cv.csv",
#     index=False,
# )
#
# predictions.to_csv(
#     f"{OUTPUT_DIR}/cv_predictions.csv",
#     index=False,
# )
#
# context_metadata_all.to_csv(
#     f"{OUTPUT_DIR}/cv_context_encoding_metadata.csv",
#     index=False,
# )
#
# fold_summary.to_csv(
#     f"{OUTPUT_DIR}/cv_fold_data_summary.csv",
#     index=False,
# )
#
# print("Results saved.")